# Kling V3 Omni Reference-to-Video API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Kling V3 Omni (Reference-to-Video)** 模型 API。

## 模型简介

Kling V3 Omni 是快手推出的全能视频生成模型，支持以下能力：

- **多模态参考生成视频**：使用参考图片、视频、音频指导视频生成
- **图片参考**：通过 `start_image_url`、`end_image_url`、`image_urls` 传入参考图片
- **视频参考**：通过 `video_list` 传入参考视频
- **Native 4K**：支持原生 4K 分辨率生成
- **音频生成**：支持生成同步音频和语音

**API 端点**：`fal-ai/kling-video/o3/{mode}/reference-to-video`

**Mode 选项**：`standard`（720P）、`pro`（1080P）、`4k`（Native 4K）

**注意**：V3 Omni 在 fal-ai 中使用 `o3` 作为模型标识

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

## 2. 基础用法 — 图片参考生成视频

七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)


# 基础调用示例：图片参考
result = run_fal_task(
    "fal-ai/kling-video/o3/standard/reference-to-video",
    arguments={
        "prompt": "从白天平滑过渡到夜晚，城市灯光逐渐亮起",
        "start_image_url": "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg",
        "end_image_url": "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg",
    },
)

# 打印返回结果
print(result)

In [ ]:
# 展示生成的视频
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 描述视频动作和运动的提示词 |
| `start_image_url` | string | 可选 | 起始帧图片 URL |
| `end_image_url` | string | 可选 | 结束帧图片 URL |
| `image_urls` | list[string] | 可选 | 多图参考 URL 列表 |
| `duration` | string | `"5"` | 视频时长：`"5"` 或 `"10"` 秒 |
| `aspect_ratio` | string | `"16:9"` | 宽高比：`"16:9"`、`"9:16"`、`"1:1"` |
| `generate_audio` | boolean | `true` | 是否生成与视频同步的音频 |
| `shot_type` | string | 可选 | 镜头类型，使用 `multi_prompt` 时必填 |
| `multi_prompt` | list | 可选 | 多镜头提示词列表 |

## 3. 多图参考生成视频

传入多张参考图片，让模型综合多个图片的内容生成视频。

In [ ]:
# 多图参考生成视频
result_multi_image = run_fal_task(
    "fal-ai/kling-video/o3/pro/reference-to-video",
    arguments={
        "prompt": "让<<<image_1>>>中的人物向背景为从<<<image_2>>>到<<<image_3>>>的场景下朝镜头挥手",
        "image_urls": [
            "https://p2-kling.klingai.com/kcdn/cdn-kcdn112452/kling-qa-test/multi-1.png",
            "https://v3b.fal.media/files/b/rabbit/NaslJIC7F2WodS6DFZRRJ.png",
            "https://v3b.fal.media/files/b/tiger/BwHi22qoQnqaTNMMhe533.png"
        ],
        "duration": "10",
    },
)

# 展示结果
video_url = result_multi_image["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. 视频参考生成视频

通过 `video_list` 传入参考视频，模型会参考视频的风格、运动方式等生成新视频。

In [ ]:
# 视频参考生成视频
result_video_ref = run_fal_task(
    "fal-ai/kling-video/o3/pro/reference-to-video",
    arguments={
        "prompt": "生成类似风格的新视频",
        "video_list": ["https://v1-kling.klingai.com/kcdn/cdn-kcdn112452/kling-qa-test/animals-output-5s.mp4"],
        "duration": "10",
    },
)

# 展示结果
video_url = result_video_ref["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. 高级用法

使用更多可选参数来精细控制视频生成效果。

In [ ]:
# 高级用法：使用更多参数
result_advanced = run_fal_task(
    "fal-ai/kling-video/o3/pro/reference-to-video",
    arguments={
        "prompt": "让<<<image_1>>>中的人物向背景为从<<<image_2>>>到<<<image_3>>>的场景下朝镜头挥手",
        "image_urls": [
            "https://p2-kling.klingai.com/kcdn/cdn-kcdn112452/kling-qa-test/multi-1.png",
            "https://v3b.fal.media/files/b/rabbit/NaslJIC7F2WodS6DFZRRJ.png",
            "https://v3b.fal.media/files/b/tiger/BwHi22qoQnqaTNMMhe533.png"
        ],
        "duration": "10",
        "shot_type": "customize",
        "aspect_ratio": "16:9"

    },
)

# 展示结果
video_url = result_advanced["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 6. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "fal-ai/kling-video/o3/standard/reference-to-video",
    arguments={
        "prompt": "参考图片生成动态视频",
        "start_image_url": "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg",
        "aspect_ratio": "16:9",
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "fal-ai/kling-video/o3/standard/reference-to-video",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 秒查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "fal-ai/kling-video/o3/standard/reference-to-video",
    request_id,
)

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 7. Schema 参考

### Input Schema

```json
{
  "prompt": "string (必填)",
  "start_image_url": "string (可选, 起始帧图片 URL)",
  "end_image_url": "string (可选, 结束帧图片 URL)",
  "image_urls": ["string (可选, 多图参考 URL 列表)"],
  "duration": "5 | 10 (可选, 默认 5)",
  "aspect_ratio": "16:9 | 9:16 | 1:1 (可选, 默认 16:9)",
  "generate_audio": "boolean (可选, 默认 true)",
  "shot_type": "string (可选, 使用 multi_prompt 时必填)",
  "multi_prompt": ["object (可选, 多镜头提示词列表)"]
}
```

### Output Schema

```json
{
  "video": {
    "file_name": "string",
    "url": "string",
    "content_type": "string",
    "file_size": "number"
  }
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [Kling V3 Omni 模型页面](https://fal.ai/models/fal-ai/kling-video/o3/standard/reference-to-video)